# Patch-level Anomaly Detection + Defect-Type Classification (실험, v2)

PatchCore + AnomalyDINO 레시피 반영으로 **안정화**:
- **coreset 메모리**(밀도 아닌 커버리지) — 임계 붕괴 방지
- **kNN 평균 anomaly**(1-NN max 아님) — 출렁임 완화
- **정상 메모리를 SUPPORT_N 에서 분리(고정)** — few-shot 은 '불량 예시 수'만 제어
- **top-1% patch 평균**으로 이미지 점수 · **detection AUROC(threshold-free)** 진단
- **(옵션) 정상 메모리 augmentation**(gamma/blur/noise) — nuisance 강건
- support/query 자동 분리 → 오염 없음

> 서버(GPU), 백본 직접.

## 0. 임포트

In [ ]:
import os, sys
from pathlib import Path
_HERE = Path.cwd(); _REPO = _HERE
while not (_REPO / 'dino_v3').exists() and _REPO != _REPO.parent:
    _REPO = _REPO.parent
for p in (str(_REPO), str(_REPO / 'dino_v3')):
    if p not in sys.path:
        sys.path.insert(0, p)
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter
import torch
import dinov3.distributed as distributed
from dinov3.eval.setup import setup_and_build_model
from inspection.em_aug import build_em_eval_transform
from sklearn.metrics import roc_auc_score
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 1. 설정

In [ ]:
CONFIG      = 'dino_v3/dinov3/configs/train/weaksup/stage2_ssl_weaksup.yaml'  # FILL_IN
CKPT        = '/path/to/teacher_checkpoint.pth'   # FILL_IN
DATA_ROOT   = '/path/to/class_folders'            # FILL_IN (classA/ ... + normal/)
NORMAL_CLASS= 'normal'

# few-shot: 불량 support 수만 제어 (정상 메모리와 무관)
SUPPORT_N   = 10
N_SEEDS     = 3            # 정량평가 반복(불량 support 재분할)

# 정상 메모리 (SUPPORT_N 과 분리, 고정)
NORMAL_MEM_FRAC = 0.7      # 정상 이미지 중 메모리용 비율(나머지=eval/임계보정)
USE_CORESET = True
CORESET_SIZE      = 1200   # coreset 후 정상 patch 수
CORESET_INPUT_CAP = 12000  # coreset 입력 상한(속도)
K_NN        = 3            # anomaly = top-K 정상 cosine 평균거리
TOPP        = 1.0          # 이미지 anomaly = 상위 TOPP% patch 평균

# 정상 메모리 augmentation (옵션)
AUG_MEMORY  = False        # True 면 정상 메모리에 gamma/blur/noise view 추가
N_AUG       = 2

IMAGE_SIZE  = 448
N_BLOCKS    = 1
MAX_IMG_PER_CLASS = 60
ANOM_PCTL   = 99.0         # bank 필터/patch gate 임계(정상 eval patch 퍼센타일)
DET_PCTL    = 95.0         # 이미지 정상/불량 판정 임계(정상 eval 이미지 퍼센타일)
CONFIG = str((_REPO / CONFIG)) if not os.path.isabs(CONFIG) else CONFIG
RNG_FIXED = np.random.default_rng(999)   # 정상 분할·coreset 고정용

## 2. 백본 + patch 추출기 + PIL augmentation

In [ ]:
if not distributed.is_enabled():
    distributed.enable(overwrite=True)
model, ctx = setup_and_build_model(config_file=CONFIG, pretrained_weights=CKPT, output_dir='./.em_cache')
model.cuda().eval()
autocast_dtype = ctx['autocast_dtype']
eval_tf = build_em_eval_transform(IMAGE_SIZE)

@torch.inference_mode()
def patch_features(pil):
    x = eval_tf(pil.convert('RGB')).unsqueeze(0).cuda()
    with torch.autocast('cuda', dtype=autocast_dtype):
        outs = model.get_intermediate_layers(x, n=N_BLOCKS, reshape=True, return_class_token=True, norm=True)
    grid = outs[-1][0][0].float().cpu().numpy()
    C, h, w = grid.shape
    f = grid.reshape(C, h * w).T
    f = f / (np.linalg.norm(f, axis=1, keepdims=True) + 1e-8)
    return f.astype(np.float32), h, w

# torch-free PIL augmentation (밝기=gamma[비선형], blur, noise). 선형밝기는 instance-norm에 상쇄되므로 gamma 사용.
def aug_view(pil):
    im = pil.convert('RGB')
    g = float(RNG_FIXED.uniform(0.7, 1.4))
    lut = [int(255 * ((v / 255.0) ** g)) for v in range(256)] * 3
    im = im.point(lut)
    r = float(RNG_FIXED.uniform(0.0, 1.5))
    if r > 0.2: im = im.filter(ImageFilter.GaussianBlur(r))
    s = float(RNG_FIXED.uniform(0.0, 0.03))
    if s > 0.005:
        arr = np.asarray(im).astype(np.float32)
        arr = arr + RNG_FIXED.normal(0.0, s * 255.0, arr.shape)
        im = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    return im

IMG_EXTS = {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def list_imgs(folder):
    return [p for p in sorted(Path(folder).rglob('*')) if p.suffix.lower() in IMG_EXTS][:MAX_IMG_PER_CLASS]

## 3. feature 사전 추출 (원본 이미지, 클래스별)
이후 split/bank/평가는 numpy. 정상 메모리는 여기서 고정 분할로 따로 만든다.

In [ ]:
root = Path(DATA_ROOT)
class_dirs = sorted([d for d in root.iterdir() if d.is_dir()])
assert any(d.name == NORMAL_CLASS for d in class_dirs), f'{NORMAL_CLASS} 폴더 필요'
defect_names = [d.name for d in class_dirs if d.name != NORMAL_CLASS]

data = {}
for d in class_dirs:
    lst = []
    for p in list_imgs(d):
        f, h, w = patch_features(Image.open(p))
        lst.append(dict(path=str(p), feats=f, h=h, w=w))
    data[d.name] = lst
    print(f'  {d.name}: {len(lst)}장')
print('defect classes:', defect_names)

## 4. 정상 메모리(coreset) + 임계 — **SUPPORT_N 과 분리, 고정**
정상 이미지를 mem/eval 로 고정 분할(seed 999). mem → coreset 메모리. eval → 임계 보정.

In [ ]:
def coreset(feats, size, seed=0):
    n = len(feats)
    if n <= size: return feats
    r = np.random.default_rng(seed)
    if n > CORESET_INPUT_CAP:
        feats = feats[r.choice(n, CORESET_INPUT_CAP, replace=False)]; n = len(feats)
    sel = [int(r.integers(n))]
    maxsim = feats @ feats[sel[0]]
    for _ in range(size - 1):
        i = int(np.argmin(maxsim)); sel.append(i)
        maxsim = np.maximum(maxsim, feats @ feats[i])
    return feats[np.array(sel)]

# 정상 mem/eval 고정 분할
n_norm = len(data[NORMAL_CLASS])
perm = RNG_FIXED.permutation(n_norm)
n_mem = max(1, int(n_norm * NORMAL_MEM_FRAC))
mem_items = [data[NORMAL_CLASS][i] for i in perm[:n_mem]]
eval_items = [data[NORMAL_CLASS][i] for i in perm[n_mem:]] or mem_items

mem_feats = [it['feats'] for it in mem_items]
if AUG_MEMORY:
    for it in mem_items:
        for _ in range(N_AUG):
            f, _, _ = patch_features(aug_view(Image.open(it['path'])))
            mem_feats.append(f)
mem_feats = np.vstack(mem_feats)
M_normal = coreset(mem_feats, CORESET_SIZE, seed=0) if USE_CORESET else mem_feats[RNG_FIXED.choice(len(mem_feats), min(CORESET_SIZE, len(mem_feats)), replace=False)]
print(f'정상 메모리: mem patch {len(mem_feats)} → 사용 {len(M_normal)} (coreset={USE_CORESET}, aug={AUG_MEMORY})')

def anom(qf):
    """top-K 정상 cosine 평균거리 (kNN anomaly). 클수록 이상."""
    out = np.empty(len(qf), np.float32)
    for i in range(0, len(qf), 2048):
        s = qf[i:i+2048] @ M_normal.T
        k = min(K_NN, s.shape[1])
        topk = np.partition(s, -k, axis=1)[:, -k:]
        out[i:i+2048] = 1.0 - topk.mean(axis=1)
    return out

def image_anom(f):
    a = anom(f); k = max(1, int(len(a) * TOPP / 100))
    return float(np.sort(a)[-k:].mean())

eval_patch_anom = anom(np.vstack([it['feats'] for it in eval_items]))
thr = float(np.percentile(eval_patch_anom, ANOM_PCTL))     # patch gate/bank 필터
det_thr = float(np.percentile([image_anom(it['feats']) for it in eval_items], DET_PCTL))  # 이미지 판정
print(f'patch thr={thr:.4f}  image det_thr={det_thr:.4f}  (정상 eval 기준, 고정)')

## 5. 진단 — detection AUROC (threshold-free) + anomaly 분포
**핵심**: 정상 vs 불량 이미지가 anomaly 점수로 갈리나? (over/under 원인이 임계인지 feature인지 규명)

In [ ]:
scores, labels = [], []
for c in data:
    items = eval_items if c == NORMAL_CLASS else data[c]
    for it in items:
        scores.append(image_anom(it['feats'])); labels.append(0 if c == NORMAL_CLASS else 1)
scores = np.array(scores); labels = np.array(labels)
auroc = roc_auc_score(labels, scores)
print(f'이미지 detection AUROC (정상 vs 불량) = {auroc:.3f}   (1.0=완벽, 0.5=무작위)')

plt.figure(figsize=(7, 4))
plt.hist(scores[labels==0], bins=30, alpha=0.6, label='normal', density=True)
plt.hist(scores[labels==1], bins=30, alpha=0.6, label='defect', density=True)
plt.axvline(det_thr, color='k', ls='--', label=f'det_thr({DET_PCTL}%)')
plt.xlabel('image anomaly (top-{}% patch 평균)'.format(TOPP)); plt.ylabel('density'); plt.legend()
plt.title(f'anomaly 분포 — AUROC={auroc:.3f}'); plt.tight_layout(); plt.show()
print('해석: 두 분포가 잘 갈리고 AUROC 높으면 → 탐지 신호 있음(임계만 잘 잡으면 됨).')
print('       겹치고 AUROC 낮으면 → feature 한계 → 학습/mask(MIL 등) 필요.')

## 6. 불량종류 bank (support, 정상대조 필터) + patch 판정

In [ ]:
def build_banks(seed):
    r = np.random.default_rng(seed)
    banks, qry = [], {}
    for c in defect_names:
        idx = r.permutation(len(data[c]))
        k = min(SUPPORT_N, max(1, len(idx) // 2))
        sup = [data[c][i] for i in idx[:k]]; qry[c] = [data[c][i] for i in idx[k:]]
        f = np.vstack([it['feats'] for it in sup]); a = anom(f)
        banks.append((c, f[a > thr]))
    return banks, qry

def classify_patches(qf, banks):
    best = np.full(len(qf), -1, int); bestsim = np.full(len(qf), -1.0, np.float32)
    for ci, (_, arr) in enumerate(banks):
        if len(arr) == 0: continue
        s = (qf @ arr.T).max(axis=1)
        m = s > bestsim; best[m] = ci; bestsim[m] = s[m]
    return best, bestsim

def analyze(item, banks):
    f = item['feats']; a = anom(f); is_anom = a > thr
    ttype, tsim = classify_patches(f, banks)
    type_map = np.where(is_anom, ttype, -1)
    if image_anom(f) < det_thr:
        pred = NORMAL_CLASS
    else:
        mass = {nm: float(tsim[is_anom & (ttype == ci)].sum()) for ci, (nm, _) in enumerate(banks)}
        pred = max(mass, key=mass.get) if any(v > 0 for v in mass.values()) else NORMAL_CLASS
    return dict(h=item['h'], w=item['w'], anom=a, is_anom=is_anom, type_map=type_map, pred=pred,
                names=[nm for nm, _ in banks])

banks0, qry0 = build_banks(0)
print('bank 등록:', {nm: len(arr) for nm, arr in banks0})

## 7. 시각화 — query 이미지 (입력 | anomaly | 종류)

In [ ]:
CLASS_COLORS = plt.cm.tab10(np.arange(len(defect_names)) % 10)[:, :3]
def show(item, r):
    h, w = r['h'], r['w']; names = r['names']
    pil = Image.open(item['path'])
    base = np.asarray(pil.resize((IMAGE_SIZE, IMAGE_SIZE)).convert('L'), float) / 255
    am = np.asarray(Image.fromarray(r['anom'].reshape(h, w).astype(np.float32)).resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR))
    tm = r['type_map'].reshape(h, w)
    over = np.zeros((h, w, 4), np.float32)
    for ci in range(len(names)):
        over[tm == ci, :3] = CLASS_COLORS[ci]; over[tm == ci, 3] = 0.8
    over_img = np.asarray(Image.fromarray((over*255).astype(np.uint8)).resize((IMAGE_SIZE, IMAGE_SIZE), Image.NEAREST)).astype(float)/255
    fig, ax = plt.subplots(1, 3, figsize=(13, 4.3))
    ax[0].imshow(base, cmap='gray'); ax[0].set_title(f'input\npred={r["pred"]}'); ax[0].axis('off')
    im = ax[1].imshow(am, cmap='inferno'); ax[1].set_title(f'anomaly (thr={thr:.3f})'); ax[1].axis('off'); fig.colorbar(im, ax=ax[1], fraction=0.046)
    ax[2].imshow(base, cmap='gray'); ax[2].imshow(over_img); ax[2].set_title('defect type'); ax[2].axis('off')
    ax[2].legend(handles=[plt.Line2D([0],[0],marker='s',ls='',color=CLASS_COLORS[ci],label=names[ci]) for ci in range(len(names))], fontsize=7, loc='upper right')
    plt.tight_layout(); plt.show()

for c in defect_names:
    for it in qry0[c][:2]:
        show(it, analyze(it, banks0))
for it in eval_items[:2]:
    show(it, analyze(it, banks0))

## 8. ✅ 확정 파이프라인 — 탐지(Youden 게이트) + 종류(logreg descriptor)

§9 진단으로 확정된 working 방식으로 end-to-end 평가한다.
- **게이트**: image anomaly > **Youden 최적 det_thr** (정상 eval + 불량 support 로 보정, query 미사용)
- **종류**: 불량 영역 **descriptor(상위-이상 patch feature 평균) → logreg** (NN-bank 아님)
- query(정상 eval + 불량 query)로 **정상+종류 end-to-end 정확도** + 불량 탐지율(recall) + 혼동행렬.

> §7 overlay 는 NN-bank(국소화 참고용), **최종 판정은 이 셀의 logreg** 이다.

In [ ]:
from sklearn.linear_model import LogisticRegression

def descriptor(item):
    f = item['feats']; a = anom(f); k = max(1, int(len(a) * TOPP / 100))
    return f[np.argsort(a)[-k:]].mean(axis=0)   # 상위-이상 patch feature 평균 = 불량영역 대표

all_names = [NORMAL_CLASS] + defect_names
accs, det_recalls = [], []
conf = np.zeros((len(all_names), len(all_names)), int)
for seed in range(N_SEEDS):
    r = np.random.default_rng(seed)
    sup, qry = {}, {}
    for c in defect_names:
        idx = r.permutation(len(data[c])); k = min(SUPPORT_N, max(1, len(idx) // 2))
        sup[c] = [data[c][i] for i in idx[:k]]; qry[c] = [data[c][i] for i in idx[k:]]
    # (a) 종류 logreg — 불량 support descriptor
    Xs = np.vstack([descriptor(it) for c in defect_names for it in sup[c]])
    ys = np.array([ci for ci, c in enumerate(defect_names) for _ in sup[c]])
    clf = LogisticRegression(max_iter=2000, class_weight='balanced').fit(Xs, ys)
    # (b) 게이트 Youden det_thr — 정상 eval + 불량 support (query 미사용)
    sc = [image_anom(it['feats']) for it in eval_items]; lb = [0] * len(eval_items)
    for c in defect_names:
        for it in sup[c]: sc.append(image_anom(it['feats'])); lb.append(1)
    sc = np.array(sc); lb = np.array(lb); tset = np.unique(sc)
    jv = [np.mean(sc[lb == 1] > t) - np.mean(sc[lb == 0] > t) for t in tset]
    dthr = float(tset[int(np.argmax(jv))])
    # (c) end-to-end 예측 (query)
    def predict(it, _clf=clf, _dthr=dthr):
        if image_anom(it['feats']) < _dthr: return NORMAL_CLASS
        return defect_names[int(_clf.predict(descriptor(it)[None])[0])]
    ok = tot = dhit = dtot = 0
    for it in eval_items:
        p = predict(it); conf[0, all_names.index(p)] += 1; ok += int(p == NORMAL_CLASS); tot += 1
    for c in defect_names:
        ci = all_names.index(c)
        for it in qry[c]:
            p = predict(it); conf[ci, all_names.index(p)] += 1
            ok += int(p == c); tot += 1; dtot += 1; dhit += int(p != NORMAL_CLASS)
    accs.append(ok / tot); det_recalls.append(dhit / max(1, dtot))
    print(f'  seed {seed}: end-to-end acc={accs[-1]*100:.1f}%  (불량탐지율 {det_recalls[-1]*100:.0f}%, det_thr={dthr:.3f})')

print(f'\nend-to-end (정상+종류) 정확도: {np.mean(accs)*100:.1f}% ± {np.std(accs)*100:.1f}%')
print(f'불량 탐지율(recall): {np.mean(det_recalls)*100:.1f}%')

fig, ax = plt.subplots(figsize=(1.1*len(all_names)+2, 1.1*len(all_names)+1))
ax.imshow(conf, cmap='Blues')
ax.set_xticks(range(len(all_names))); ax.set_xticklabels(all_names, rotation=45, ha='right')
ax.set_yticks(range(len(all_names))); ax.set_yticklabels(all_names)
ax.set_xlabel('pred'); ax.set_ylabel('true'); ax.set_title('end-to-end confusion (query)')
for i in range(len(all_names)):
    for j in range(len(all_names)): ax.text(j, i, conf[i, j], ha='center', va='center')
plt.tight_layout(); plt.show()

## 참고 / 튜닝
- **정상 메모리는 SUPPORT_N 과 무관하게 고정** → SUPPORT_N 바꿔도 anomaly 스케일 안정.
- **§5 AUROC** 가 핵심 진단: 높으면 탐지 신호 있음(임계 문제였음), 낮으면 feature 한계(→학습/mask).
- 노브: `K_NN`(평균 이웃), `CORESET_SIZE`, `TOPP`(이미지 집계), `ANOM_PCTL`/`DET_PCTL`(임계),
  `AUG_MEMORY`(정상 nuisance 강건), `N_BLOCKS`(레이어; 중간층은 get_intermediate_layers(n=[3,6,9])).
- 종류 분류가 안 되면: rough mask 로 defect patch 직접 지정 / MIL(attention) / 백본 weak-sup 재학습.

## 9. typing 진단 — **불량 이미지에만, 게이트 우회** (NN vs 판별 logreg)

§8 은 `image_anom < det_thr → normal` 게이트 때문에 불량이 다 normal 로 샐 수 있다(typing 이 가려짐).
여기선 **불량 query 이미지에만** 게이트 없이 '무슨 종류냐'만 평가한다.
- **NN-bank**: 이상 patch → 가장 가까운 불량 bank (현재 방식)
- **logreg(descriptor)**: 이미지의 **상위-이상 patch feature 평균**(불량 영역 대표) → 불량종류 logreg

logreg 가 되면 → feature 엔 종류 신호 있음(NN 집계가 약했던 것). 둘 다 낮으면 → feature 한계(mask/재학습).

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def defect_descriptor(item):
    f = item['feats']; a = anom(f); k = max(1, int(len(a) * TOPP / 100))
    idx = np.argsort(a)[-k:]        # 상위-이상 patch
    return f[idx].mean(axis=0)

# 참고: 게이트(det_thr)의 Youden 최적점 — '다 normal' 이 임계 탓인지 확인
scr, lab = [], []
for c in data:
    for it in (eval_items if c == NORMAL_CLASS else data[c]):
        scr.append(image_anom(it['feats'])); lab.append(0 if c == NORMAL_CLASS else 1)
scr = np.array(scr); lab = np.array(lab)
ts = np.sort(np.unique(scr))
j = [ (np.mean(scr[lab==1] > t) - np.mean(scr[lab==0] > t)) for t in ts ]
best_t = float(ts[int(np.argmax(j))])
print(f'현재 det_thr={det_thr:.4f} → 불량 통과율 {np.mean(scr[lab==1]>det_thr)*100:.0f}% (다 normal 이면 여기가 낮음)')
print(f'Youden 최적 det_thr={best_t:.4f} → 불량 통과율 {np.mean(scr[lab==1]>best_t)*100:.0f}%, 정상 오탐 {np.mean(scr[lab==0]>best_t)*100:.0f}%')

# defect-only typing (게이트 우회)
nn_accs, lr_accs = [], []
conf = np.zeros((len(defect_names), len(defect_names)), int)   # true x pred (NN-bank)
for seed in range(N_SEEDS):
    banks, qry = build_banks(seed)
    r = np.random.default_rng(seed)   # build_banks 와 동일 분할 재현
    Xs, ys = [], []
    for ci, c in enumerate(defect_names):
        idx = r.permutation(len(data[c])); k = min(SUPPORT_N, max(1, len(idx) // 2))
        for i in idx[:k]:
            Xs.append(defect_descriptor(data[c][i])); ys.append(ci)
    Xs = np.vstack(Xs); ys = np.array(ys)
    clf = LogisticRegression(max_iter=2000, class_weight='balanced').fit(Xs, ys) if len(set(ys)) > 1 else None
    okn = okl = tot = 0
    for ci, c in enumerate(defect_names):
        for it in qry[c]:
            f = it['feats']; a = anom(f); is_anom = a > thr
            tt, tsim = classify_patches(f, banks)
            sel = is_anom if is_anom.any() else np.ones(len(f), bool)
            mass = [float(tsim[sel & (tt == kk)].sum()) for kk in range(len(defect_names))]
            pn = int(np.argmax(mass))
            conf[ci, pn] += 1; okn += int(pn == ci)
            if clf is not None:
                okl += int(int(clf.predict(defect_descriptor(it)[None])[0]) == ci)
            tot += 1
    nn_accs.append(okn / max(1, tot)); lr_accs.append(okl / max(1, tot))

print(f'\n[defect-only typing, 게이트 우회]  (클래스 수={len(defect_names)}, chance≈{100/len(defect_names):.0f}%)')
print(f'  NN-bank            : {np.mean(nn_accs)*100:.1f}% ± {np.std(nn_accs)*100:.1f}%')
print(f'  logreg(descriptor) : {np.mean(lr_accs)*100:.1f}% ± {np.std(lr_accs)*100:.1f}%')

fig, ax = plt.subplots(figsize=(1.1*len(defect_names)+2, 1.1*len(defect_names)+1))
ax.imshow(conf, cmap='Blues')
ax.set_xticks(range(len(defect_names))); ax.set_xticklabels(defect_names, rotation=45, ha='right')
ax.set_yticks(range(len(defect_names))); ax.set_yticklabels(defect_names)
ax.set_xlabel('pred'); ax.set_ylabel('true'); ax.set_title('NN-bank typing confusion (defect-only)')
for i in range(len(defect_names)):
    for jj in range(len(defect_names)): ax.text(jj, i, conf[i, jj], ha='center', va='center')
plt.tight_layout(); plt.show()

print('\n해석:')
print('  logreg ≫ NN-bank 이고 logreg 높음 → feature 엔 종류신호 O, NN 집계가 약했던 것 → 판별 head 로 교체')
print('  둘 다 chance 근처       → feature 한계 → rough mask(patch 지도)/백본 weak-sup 재학습 필요')
print('  게이트: Youden det_thr 로 바꾸면 §8 의 다-normal 은 완화(탐지 임계 문제였음)')